In [ ]:
from pathlib import Path
import sys

src_dir = next(p for p in [Path.cwd(), *Path.cwd().parents]
               if (p / "common.py").is_file() and (p / "feature_preference").is_dir())
sys.path.insert(0, str(src_dir))


In [ ]:
import torch
import torch.nn as nn
from tqdm.auto import tqdm
import numpy as np
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
import random

from common import *
import yaml
import matplotlib.colors as mcolors

import library

In [ ]:
device = torch.device("cuda")
sns.set_theme(style="whitegrid", font="cmr10", font_scale=1.9)
plt.rcParams["axes.formatter.use_mathtext"] = True
plt.rcParams.update({"pdf.fonttype": 42, "ps.fonttype": 42})
plt.rcParams.update({
    'text.usetex': False,
    'mathtext.fontset': 'cm',  # Computer Modern math font
})
colors = sns.color_palette()
colors

default plot setting

In [ ]:
with open("../linear/config.yaml", 'r', encoding='utf-8') as f:
    configs_linear = yaml.safe_load(f)
with open("../deep/config.yaml", 'r', encoding='utf-8') as f:
    configs_deep = yaml.safe_load(f)
with open("../mlp/config.yaml", 'r', encoding='utf-8') as f:
    configs_mlp = yaml.safe_load(f)

config = configs_deep

In [ ]:
def sample(batch, sizes, device, mul=0.01):
    x_out = torch.zeros((batch, 0), device=device)
    for size in sizes:
        rand = torch.rand((batch, 1), device=device).round() * 2 - 1
        x_out = torch.cat([x_out, rand.repeat(1, size)], dim=-1)
    epsilon1 = mul * torch.randn_like(x_out)
    epsilon2 = mul * torch.randn_like(x_out)
    return x_out + epsilon1, x_out + epsilon2

sample makes [+1, -1] datas

In [ ]:
def create_segment_vectors(sizes):
    total_length = sum(sizes)
    cumsum = torch.cumsum(torch.tensor(sizes), dim=-1)

    result = torch.zeros((len(sizes), total_length))

    for i in range(len(sizes)):
        start_idx = cumsum[i - 1] if i > 0 else 0
        end_idx = cumsum[i]

        result[i, start_idx:end_idx] = 1

    return result

In [ ]:
batch = config["batch"]
sizes = config["sizes"]
s_in = sum(sizes)
s_out = 2

In [ ]:
sample_data = sample(batch, sizes, device)
gamma = torch.bmm(sample_data[0].unsqueeze(dim=2), sample_data[1].unsqueeze(dim=1))  # b x n x n
gamma = gamma + gamma.permute(0, 2, 1)
gamma = torch.mean(gamma, dim=0).squeeze() / 2  # n x n
sns.heatmap(gamma.cpu())
plt.show()
gamma_l, gamma_v = torch.linalg.eig(gamma)
gamma_l = gamma_l.real.cpu()
gamma_v = gamma_v.real.cpu()
print(gamma_l, gamma_v)

In [ ]:
output_dict = np.load("data.npy", allow_pickle=True).item()

In [ ]:
loss_types = "BarlowTwins_balanced"

e = create_segment_vectors(sizes).to(device)

# lambdas_render_barlow = ["0", "0.1", "0.5", "1"]
lambdas_render_barlow = ["1"]

In [ ]:
L = 4
power_value = 3 - 2 / L


def dsdt0(t, s):
    gamma = gamma_l[0].item()
    return 4 * L * (s ** power_value) * gamma * (1 - (s ** 2) * gamma)


def dsdt1(t, s):
    gamma = gamma_l[1].item()
    return 4 * L * (s ** power_value) * gamma * (1 - (s ** 2) * gamma)


In [ ]:
from scipy.integrate import solve_ivp

t0 = 13000
epochs = config[loss_types]["epochs"]

lambd = "1"
if "BarlowTwins" in loss_types:
    output_target = output_dict[loss_types][lambd]
else:
    output_target = output_dict[loss_types]
losses = output_target["loss"]
eiglist = output_target["eiglist"]
feature_alignment = output_target["feature_alignment"]
s0 = output_target["s0"]

dldt = smooth_data(dt(losses), 15)
plt.plot(dldt)

fig, axs = plt.subplots(1, 3, figsize=(20, 4))

axs[0].plot(losses, color=colors[2], linewidth=6, alpha=0.5)
critical_pos = []
starting = 0
for i in range(len(eiglist)):
    find = np.argmin(dldt[starting:])
    if dldt[starting + find] < -0.001:
        critical_pos.append(starting + find)
        starting += find + 200
    else:
        break

if loss_types == "BarlowTwins":
    axs[0].set_ylim(-0.05, s_out * 1.15)
for pos in critical_pos:
    axs[0].axvline(pos, color='black', linestyle='--')
axs[0].set_xlabel("Step")
axs[0].set_ylabel("Loss")

for i in range(len(eiglist)):
    axs[1].plot(eiglist[i], linewidth=6, color=colors[i], alpha=0.5,
                label=(r"$\lambda_" + ("l$" if i == 0 else "s$")))
for pos in critical_pos:
    axs[1].axvline(pos, color='black', linestyle='--')
# axs[1].set_ylim(-0.05, 1.10)
axs[1].set_xlabel("Step")
axs[1].set_ylabel(r"C Eigenvalue $\lambda$")
axs[1].legend()

for axis in axs:
    axis.set_xlim(-epochs * 0.02, epochs * 1.02)

if config["network"] != "mlp":
    if loss_types == "BarlowTwins_balanced":
        t1 = 50
        predict_range = 800
        t_common = np.linspace(0, predict_range, predict_range * 5)
        sol0 = solve_ivp(dsdt0, (0, predict_range), (s0[0].item(),), rtol=1e-8, atol=1e-8, t_eval=t_common)
        sol1 = solve_ivp(dsdt1, (0, predict_range), (s0[1].item(),), rtol=1e-8, atol=1e-8, t_eval=t_common)
        y1 = np.pow(sol0.y[0], 2) * gamma_l[0].item()
        y2 = np.pow(sol1.y[0], 2) * gamma_l[1].item()
        predicted_loss = (y1 - 1) ** 2 + (y2 - 1) ** 2
        t_range = t1 * sol0.t
        axs[1].plot(t_range, y1, linestyle='--', color=colors[0], linewidth=2.5)
        axs[1].plot(t_range, y2, linestyle='--', color=colors[1], linewidth=2.5)
        axs[2].plot(t_range, math.sqrt(sizes[0]) * sol0.y[0], linestyle='--', color=colors[0], linewidth=2.5)
        axs[2].plot(t_range, math.sqrt(sizes[1]) * sol1.y[0], linestyle='--', color=colors[1], linewidth=2.5)
        t2 = 10000
        # axs[1].axhline((3 - 2 / L) / (5 - 2 / L), color='blue')
        # axs[1].axvline((4 - 2 / L) / (gamma_l[1].item() * (5 - 2 / L)) * t2, color='blue')
        axs[1].set_yscale('log')

        axs[0].plot(t_range, predicted_loss, linestyle='--', color=colors[2], linewidth=2.5)

    for i in range(len(feature_alignment)):
        k = np.array(feature_alignment[i])
        axs[2].plot(k, label=(r"${\text{FA}(e_l)}$" if i == 0 else r"${\text{FA}(e_s)}$"), alpha=0.5,
                    linewidth=6, color=colors[i])
    for pos in critical_pos:
        axs[2].axvline(pos, color='black', linestyle='--')
    # axs[2].set_ylim(-0.05, 1.10)
    axs[2].legend()
    axs[2].set_xlabel("Step")
    axs[2].set_ylabel(r"$\|\tilde We_i\|_2$")
print(critical_pos)

plt.tight_layout()
plt.savefig("figure/balanced/dim_loss_eig_align.pdf", format="pdf", bbox_inches='tight')
plt.show()

In [ ]:
from scipy.integrate import solve_ivp

t0 = 13000
epochs = config[loss_types]["epochs"]

lambd = "1"
if "BarlowTwins" in loss_types:
    output_target = output_dict[loss_types][lambd]
else:
    output_target = output_dict[loss_types]
losses = output_target["loss"]
eiglist = output_target["eiglist"]
feature_alignment = output_target["feature_alignment"]
s0 = output_target["s0"]

fig, axs = plt.subplots(1, 1, figsize=(10, 5))

axs.plot(losses, color=colors[1], linewidth=10, alpha=0.9)
critical_pos = []
starting = 0
for i in range(len(eiglist)):
    find = np.argmin(dldt[starting:])
    if dldt[starting + find] < -0.001:
        critical_pos.append(starting + find)
        starting += find + 200
    else:
        break

if loss_types == "BarlowTwins":
    axs.set_ylim(-0.05, s_out * 1.15)
axs.set_xlabel("Step")
axs.set_ylabel("Loss")

axs.set_xlim(-0.05, 18000)
if config["network"] != "mlp":
    if loss_types == "BarlowTwins_balanced":
        t1 = 50
        predict_range = 800
        t_common = np.linspace(0, predict_range, predict_range * 5)
        sol0 = solve_ivp(dsdt0, (0, predict_range), (s0[0].item(),), rtol=1e-8, atol=1e-8, t_eval=t_common)
        sol1 = solve_ivp(dsdt1, (0, predict_range), (s0[1].item(),), rtol=1e-8, atol=1e-8, t_eval=t_common)
        y1 = np.pow(sol0.y[0], 2) * gamma_l[0].item()
        y2 = np.pow(sol1.y[0], 2) * gamma_l[1].item()
        predicted_loss = (y1 - 1) ** 2 + (y2 - 1) ** 2
        t_range = t1 * sol0.t
        t2 = 10000
        # axs[1].axhline((3 - 2 / L) / (5 - 2 / L), color='blue')
        # axs[1].axvline((4 - 2 / L) / (gamma_l[1].item() * (5 - 2 / L)) * t2, color='blue')

        axs.plot(t_range, predicted_loss, color='blue', linewidth=2.5)

# for pos in critical_pos:
#     axs.axvline(pos, color='black', linestyle='--')
print(critical_pos)
axs.set_ylim(-0.05, 2 * 1.15)
plt.tight_layout()
plt.savefig("figure/balanced/dim_loss_only.pdf", format="pdf", bbox_inches='tight')
plt.show()

In [ ]:
epochs = config[loss_types]["epochs"]
lr = float(config[loss_types]["lr"])
fig, axs = plt.subplots(1, 1, figsize=(12, 4))
lambd = "1"
print(output_dict[loss_types])
output_target = output_dict[loss_types][lambd]
losses = output_target["loss"]
eiglist = output_target["eiglist"]
feature_alignment = output_target["feature_alignment"]
s0 = output_target["s0"]
subspace_alignment = output_target["subspace_alignment"]
x = np.linspace(0, len(subspace_alignment), len(subspace_alignment))
if "log_freq" in config:
    x = x * config["log_freq"]
#||A^TB||^2/2
axs.plot(x, subspace_alignment, alpha=0.6, color=colors[index], linewidth=5,
         label=r"BT, $\lambda = " + f"{lambd}$")
axs.set_xlim(0, epochs)
axs.set_ylim(-0.05, 1.05)
axs.set_xlabel("Step")
axs.set_ylabel("Subspace Alignment")
axs.legend()
plt.savefig("figure/balanced/dim_alignment.pdf", format="pdf", bbox_inches='tight')
plt.show()